In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression

# from sklearn.naive_bayes import MultinomialNB
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import LinearSVC

In [2]:
## Loading the dfs
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')

In [3]:
# Let's fetch all the possible text data
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)

In [4]:
#Initialize models and tools
vectorizer = CountVectorizer()
tfidfzer = TfidfTransformer()
lr = LogisticRegression(penalty='l2', dual=False, tol=0.0001, C=1.0, fit_intercept=True, intercept_scaling=1, class_weight=None, random_state=42, solver='newton-cg', max_iter=100, multi_class='auto', verbose=0, warm_start=False, n_jobs=None, l1_ratio=None)

In [5]:




#Split train and test data
X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], random_state=42)


#Transform X_train
vectors = vectorizer.fit_transform(X_train)
tfidf = tfidfzer.fit_transform(vectors)

#Train model
model = lr.fit(tfidf,y_train)
from sklearn import metrics
vectors_test = vectorizer.transform(X_test)
pred = lr.predict(vectors_test)

acc_score = metrics.accuracy_score(y_test, pred)
f1_score = metrics.f1_score(y_test, pred, average='macro')
print("--------------------------------------------------")
print('  Accuracy: {}%'.format(round(acc_score*100,3)))
print('       F1 : {}%'.format(round(f1_score*100,3)))
print("--------------------------------------------------")

--------------------------------------------------
  Accuracy: 88.571%
       F1 : 88.611%
--------------------------------------------------


In [6]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, pred)

array([[71, 11,  2],
       [ 5, 72,  6],
       [ 4,  0, 74]], dtype=int64)

In [14]:
# text = ["I have a very severe and throbbing ear."]
text = "my eyes are red. it is swollen and it's painful to touch .it's been reddish since last week."
# text = 'My        are hurting after I swam in the lake.    I cannot      anymore'
location = 'Manila'


In [15]:
# Add the next text to the original text
proba = lr.predict_proba((vectorizer.transform([text])))[0]

if len(text)<70:
    print('Tell me more')

else:
    if max(proba) > 0.50:
        result = model.predict(vectorizer.transform([text]))[0]
    else:
        result = "Undecided"
        print("""I cannot recommend a specialist based on your symptoms.  
                 I suggest that you seek consult with a General Physician via telemedicine.
                 Here are the recommendations:
                 
                 Doctor's Name: Dr. Maria Bathilda I. Tampi
                 Telemedicine Link: drmabait@consult.com
                 
                 Doctor's Name: Dr. Leopold G.  Hiran
                 Telemedicine Link: nicedoctor@telemed.com
                    
              """)
print(result)

I cannot recommend a specialist based on your symptoms.  
                 I suggest that you seek consult with a General Physician via telemedicine.
                 Here are the recommendations:
                 
                 Doctor's Name: Dr. Maria Bathilda I. Tampi
                 Telemedicine Link: drmabait@consult.com
                 
                 Doctor's Name: Dr. Leopold G.  Hiran
                 Telemedicine Link: nicedoctor@telemed.com
                    
              
Undecided


In [9]:
# Saving model
filename = "CheckApp_LR_Model.pickle"  

with open(filename, 'wb') as file:  
    pickle.dump(model, file)
    
# Saving model
filename = "vectorizer.pickle"  

with open(filename, 'wb') as file:  
    pickle.dump(vectorizer, file)
    
filename = "lr.pickle"  

with open(filename, 'wb') as file:  
    pickle.dump(lr, file)

In [10]:
# Output Results
df_res = df_doc[(df_doc['Specialty']==result) & (df_doc["Clinic"].str.lower().str.contains(location))]
df_online = df_doc.replace(np.nan, '', regex=True)
df_online = df_online[(df_online['Specialty']==result) & (df_online['Online']!="")] 

In [11]:
# Print results
if len(df_res) == 0:
    print(f"""      Searched for: {result} in {location}
      None found in the list
      
      Another option is consulting via telemedicine with the following specialists:
      """)
    for index,row in df_online.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Link for Telemedicine Consult: {row["Online"]}
        """)
else:
    print(f"       Searched for: {result} in {location}")
    for index,row in df_res.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Clinic Address: {(row["Clinic"])}
            Contact Details For Appointment: {row["Contact"]}
        """)

      Searched for: Undecided in Manila
      None found in the list
      
      Another option is consulting via telemedicine with the following specialists:
      


In [12]:
df_doc

,Doctor,Specialty,Location,Clinic,Contact,Online
0,"Criscely L. Go, MD",Neurologist,Manila,"Metropolitan Medical Center Room 224, Masangka...",(02) 425 9779 / (02) 254 1111 local 2033 / (09...,NaN
1,"Dominador D. Baldonado, MD",Neurologist,Quezon City,FEU-NRMF Medical Center Room 415 Marian Medica...,(02) 461 3134 / (02) 427 0213,NaN
2,"Euvin Paul G. Lagapa, MD",Neurologist,Davao City,"San Pedro Hospital Room 307 Rita Stang Bldg., ...",(082) 221 7272,NaN
3,"Paul Matthew D. Pasco, MD",Neurologist,Manila,"Philippine General Hospital, Taft Avenue, Ermi...",(02) 8521 8450,NaN
4,"Abdias V. Aquino, MD",Neurologist,Quezon City,"National Kidney and Transplant Institute, East...",(02) 8981 0400,NaN
...,...,...,...,...,...,...
174,"Flores, Ruben",Neurologist,Iligan,NaN,drrubenflores.neurology@gmail.com,NaN
175,"Diamla, Sharimah",Neurologist,Iligan/Marawi,NaN,shahjamla@yahoo.com,NaN
176,"Ugdamina, Fritzie",Neurologist,Ozamiz City,NaN,junfritz7583@yahoo.com,NaN
177,"Yap-Domingo, Archie",Neurologist,Ozamiz City,NaN,archieyapdomingo@gmail.com,NaN
